In [3]:
import numpy as np
import torch
import torch.nn as nn

# Actividad 1: Análisis de Sensibilidad

Definimos a I 

$$
I =
\begin{bmatrix}
10 & 10 & 10 \\
10 & 80 & 10 \\
10 & 10 & 10
\end{bmatrix}
$$

Filtro Kernel 


$$
K =
\begin{bmatrix}
1 & 0 \\
0 & -1
\end{bmatrix}
$$

Este filtro detecta cambios entre el valor de la esquina superior izquierda y la esquina inferior derecha de cada ventana.


Convolucaion de la imagen 
Como el filtro es $2\times2$ y la imagen es $3\times3$, el **Feature Map es de $2\times2$**.
Lo dividimos por ventanas 
**La ventana 1**
Submatriz:

$$
\begin{bmatrix}
10 & 10 \\
10 & 80
\end{bmatrix}
$$

Operación:

$(10 \cdot 1) + (10 \cdot 0) + (10 \cdot 0) + (80 \cdot -1) = 10 - 80 = -70$

**La ventana 2**

Submatriz:

$$
\begin{bmatrix}
10 & 10 \\
80 & 10
\end{bmatrix}
$$

Operación:

$ (10 \cdot 1) + (10 \cdot 0) + (80 \cdot 0) + (10 \cdot -1) = 10 - 10 = 0 $

**La venatana 3**

Submatriz:

$$
\begin{bmatrix}
10 & 80 \\
10 & 10
\end{bmatrix}
$$

Operación:

$(10 \cdot 1) + (80 \cdot 0) + (10 \cdot 0) + (10 \cdot -1) = 10 - 10 = 0 $

**La ventana 4**

Submatriz:

$$
\begin{bmatrix}
80 & 10 \\
10 & 10
\end{bmatrix}
$$

Operación:

$ (80 \cdot 1) + (10 \cdot 0) + (10 \cdot 0) + (10 \cdot -1) = 80 - 10 = 70 $

El Feature Map original es

$$
\begin{bmatrix}
-70 & 0 \\
0 & 70
\end{bmatrix}
$$

---

Modificamos el valor central de 80 a 20

Nueva imagen:

$$
I' =
\begin{bmatrix}
10 & 10 & 10 \\
10 & 20 & 10 \\
10 & 10 & 10
\end{bmatrix}
$$

--- 

Convolucion Modifica(igual lo dividi en ventanas)

**Ventana 1**
$$
\begin{bmatrix}
10 & 10 \\
10 & 20
\end{bmatrix}



= (10 \cdot 1) + (10 \cdot 0) + (10 \cdot 0) + (20 \cdot -1) = 10 - 20 = -10
$$

**Ventana 2**

$$
\begin{bmatrix}
10 & 10 \\
20 & 10
\end{bmatrix}

= (10 \cdot 1) + (10 \cdot 0) + (20 \cdot 0) + (10 \cdot -1) = 10 - 10 = 0
$$

**Ventana 3**

$$
\begin{bmatrix}
10 & 20 \\
10 & 10
\end{bmatrix}

= (10 \cdot 1) + (20 \cdot 0) + (10 \cdot 0) + (10 \cdot -1) = 10 - 10 = 0
$$

**Ventana 4**

$$
\begin{bmatrix}
20 & 10 \\
10 & 10
\end{bmatrix}

= (20 \cdot 1) + (10 \cdot 0) + (10 \cdot 0) + (10 \cdot -1) = 20 - 10 = 10
$$

Entonces el Feature Modificado me queda como: 
$$
\begin{bmatrix}
-10 & 0 \\
0 & 10
\end{bmatrix}
$$

Como el filtro detecta cambios de intensidad, la magnitud del Feature Map también disminuye.

Esto muestra que cuando el contraste en la imagen original es menor, la activación del filtro también es menor.

In [ ]:
# Imagen original
I = np.array([
    [10, 10, 10],
    [10, 80, 10],
    [10, 10, 10]
])

# Imagen modificada
I_mod = np.array([
    [10, 10, 10],
    [10, 20, 10],
    [10, 10, 10]
])

# Filtro
K = np.array([
    [1, 0],
    [0, -1]
])

def conv2d_valid(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = h - kh + 1
    out_w = w - kw + 1
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(region * kernel)
    
    return output

feature_original = conv2d_valid(I, K)
feature_mod = conv2d_valid(I_mod, K)

print("Feature map original:")
print(feature_original)

print("\nFeature map modificado:")
print(feature_mod)

Feature map original:
[[-70.   0.]
 [  0.  70.]]

Feature map modificado:
[[-10.   0.]
 [  0.  10.]]


# Actividad 2: Exploración de Hiperparámetros

2.1 Crear un tensor que represente una imagen 3×3

Primero creamos un tensor que represente una imagen en escala de grises de tamaño $3\times3.$

En PyTorch las imágenes para convoluciones tienen la forma:

$(batch\_size, channels, height, width)$

Enotnces agregamos las dimensiones necesarias.

In [4]:
# Imagen 3x3
I = torch.tensor([
    [10., 10., 10.],
    [10., 80., 10.],
    [10., 10., 10.]
])

# Agregamos batch y canal
I = I.unsqueeze(0).unsqueeze(0)

print(I.shape)

torch.Size([1, 1, 3, 3])


Definimos el filtro 

In [5]:
K = torch.tensor([
    [1., 0.],
    [0., -1.]
])

Creamos una capa convolucional con : 
- 1 canal de entrada 
- 1 filtro 
- Kernel 2x2

In [6]:
conv = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    bias=False
)

Inicializamos manualmente los pesos 

In [7]:
with torch.no_grad():
    conv.weight[:] = K.view(1,1,2,2)

Pasamos la imagen a traves de la capa 

In [8]:
feature_map = conv(I)

print(feature_map)

tensor([[[[-70.,   0.],
          [  0.,  70.]]]], grad_fn=<ConvolutionBackward0>)


Modificar el parametro Stride 

In [9]:
conv_stride = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    stride=2,
    bias=False
)

with torch.no_grad():
    conv_stride.weight[:] = K.view(1,1,2,2)

feature_stride = conv_stride(I)

print(feature_stride)

tensor([[[[-70.]]]], grad_fn=<ConvolutionBackward0>)


Agregamos el Padding

In [10]:
conv_padding = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    padding=1,
    bias=False
)

with torch.no_grad():
    conv_padding.weight[:] = K.view(1,1,2,2)

feature_padding = conv_padding(I)

print(feature_padding)

tensor([[[[-10., -10., -10.,   0.],
          [-10., -70.,   0.,  10.],
          [-10.,   0.,  70.,  10.],
          [  0.,  10.,  10.,  10.]]]], grad_fn=<ConvolutionBackward0>)


# Actividad 3: Respuestas

## 1. Tamaño del mapa de características

La fórmula general para calcular el tamaño de salida de una convolución es:

$\text{Salida} = \frac{N - K + 2P}{S} + 1$

donde:

- $N$ = tamaño de la imagen
- $K$ = tamaño del kernel
- $P$ = padding
- $S$ = stride

En este ejercicio:

- $N = 3$
- $K = 2$
- $P = 0$
- $S = 1$

Entonces:

$\frac{3 - 2 + 0}{1} + 1 = 2$

Por lo tanto, el **mapa de características resultante tiene tamaño $2 \times 2$**.


## 2. Efecto del parámetro `bias=False`

El parámetro `bias=False` indica que **no se agregará un término de sesgo (bias)** al resultado de la convolución.

Normalmente una convolución calcula:

$$
y = (x * w) + b
$$

donde:

- $x$ es la entrada
- $w$ es el kernel
- $b$ es el bias

Al usar `bias=False`, eliminamos el término $b$, por lo que el resultado depende **únicamente de la operación de convolución**.

Esto es útil cuando queremos que el cálculo coincida exactamente con una **convolución manual**, como en este ejercicio.

## 3.Primera dimensión del tensor de pesos

En PyTorch, el tensor de pesos de una capa `Conv2d` tiene la forma:

$$
(out\_channels,\ in\_channels,\ height,\ width)
$$

La **primera dimensión (`out_channels`) representa el número de filtros** que tiene la capa.

Cada filtro produce **un mapa de características distinto**, por lo que si tenemos $n$ filtros, obtendremos $n$ feature maps en la salida.


## 4. Aplicar el kernel a una ventana con valores iguales

Kernel:

$$
K =
\begin{bmatrix}
1 & 0 \\
0 & -1
\end{bmatrix}
$$

Ventana de la imagen:

$$
\begin{bmatrix}
50 & 50 \\
50 & 50
\end{bmatrix}
$$

Operación de convolución:

$(50 \cdot 1) + (50 \cdot 0) + (50 \cdot 0) + (50 \cdot -1)= 50 - 50 = 0$

El valor resultante será **0**, ya que no existe diferencia entre los valores de la ventana.

Esto ocurre porque el filtro detecta **cambios de intensidad**, y en este caso todos los píxeles son iguales.


## 5. Uso de `torch.no_grad()`

Se utiliza para indicar a PyTorch que no calcule gradientes durante las operaciones dentro del bloque. Esto es necesario porque los pesos de una red neuronal normalmente forman parte del grafo de cálculo para backpropagation.

Cuando asignamos manualmente valores a los pesos con:
```python
conv.weight = ...
```

si no usamos `torch.no_grad()`, PyTorch intentaría registrar esa operación para el cálculo de gradientes.


Al usar torch.no_grad():

- evitamos modificar el grafo de cálculo

- prevenimos errores durante el entrenamiento

- permitimos asignar pesos manualmente de forma segura
